# 🛒 Analyse de données US Superstore — Stratégie Marketing

Ce notebook réalise une analyse complète des données de vente du **US Superstore** afin d'orienter une stratégie marketing : analyse géographique, analyse client, analyse par catégorie de produits, séries chronologiques et application du **principe de Pareto (80/20)**.

**Plan :**
1. Chargement & prétraitement des données
2. États avec le plus de ventes
3. Comparaison New York vs Californie
4. Client exceptionnel à New York
5. Différences de rentabilité entre États
6. Pareto : clients & bénéfices
7. Top 20 villes (ventes & bénéfices) + rentabilité
8. Top 20 clients par ventes
9. Courbe cumulative des ventes + Pareto clients/ventes
10. Recommandations marketing

## 1. Installation & importation des bibliothèques

In [ ]:
# xlrd est nécessaire pour lire les fichiers .xls
!pip install xlrd -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
pd.options.display.float_format = "{:,.2f}".format

### Chargement du jeu de données

Deux options :
- **Option A** — téléchargement direct depuis GitHub (recommandé sur Colab).
- **Option B** — upload manuel du fichier `US Superstore data.xls`.

In [ ]:
# --- Option A : téléchargement direct ---
url = "https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/US%20Superstore%20data.xls"
df = pd.read_excel(url)

# --- Option B : upload manuel (décommentez si besoin) ---
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_excel("US Superstore data.xls")

df.head()

## 2. Prétraitement des données

In [ ]:
# Dimensions et types
print("Dimensions :", df.shape)
df.info()

In [ ]:
# Valeurs manquantes et doublons
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())
print("\nNombre de doublons :", df.duplicated().sum())

In [ ]:
# Conversion des dates (au cas où) et création de colonnes temporelles utiles
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"]  = pd.to_datetime(df["Ship Date"])

df["Year"]        = df["Order Date"].dt.year
df["Month"]       = df["Order Date"].dt.month
df["YearMonth"]   = df["Order Date"].dt.to_period("M").astype(str)

# Marge bénéficiaire par ligne (utile pour l'analyse de rentabilité)
df["Profit Margin"] = df["Profit"] / df["Sales"]

df[["Order Date","Year","Month","YearMonth","Sales","Profit","Profit Margin"]].head()

In [ ]:
# Statistiques descriptives des variables numériques clés
df[["Sales","Quantity","Discount","Profit"]].describe()

## 3. Quels États enregistrent le plus de ventes ?

On agrège les ventes par État et on visualise le Top 15.

In [ ]:
state_sales = (df.groupby("State")["Sales"].sum()
                 .sort_values(ascending=False))

top15_states = state_sales.head(15)

plt.figure(figsize=(12,7))
sns.barplot(x=top15_states.values, y=top15_states.index, palette="viridis")
plt.title("Top 15 des États par chiffre d'affaires")
plt.xlabel("Ventes totales ($)")
plt.ylabel("État")
for i, v in enumerate(top15_states.values):
    plt.text(v, i, f" {v:,.0f}", va="center")
plt.tight_layout()
plt.show()

print("Top 5 États par ventes :")
print(state_sales.head())

**Observation :** la **Californie** et **New York** dominent largement le chiffre d'affaires, suivies du **Texas**. Ce sont des marchés prioritaires.

## 4. New York vs Californie — chiffre d'affaires et bénéfices

In [ ]:
compare = (df[df["State"].isin(["New York","California"])]
           .groupby("State")[["Sales","Profit"]].sum())
compare["Profit Margin %"] = 100 * compare["Profit"] / compare["Sales"]
compare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,6))

compare["Sales"].plot(kind="bar", ax=axes[0], color=["#4C72B0","#DD8452"])
axes[0].set_title("Chiffre d'affaires total")
axes[0].set_ylabel("Ventes ($)")
axes[0].tick_params(axis="x", rotation=0)
for i, v in enumerate(compare["Sales"]):
    axes[0].text(i, v, f"{v:,.0f}", ha="center", va="bottom")

compare["Profit"].plot(kind="bar", ax=axes[1], color=["#4C72B0","#DD8452"])
axes[1].set_title("Bénéfice total")
axes[1].set_ylabel("Bénéfice ($)")
axes[1].tick_params(axis="x", rotation=0)
for i, v in enumerate(compare["Profit"]):
    axes[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

**Observation :** la Californie génère un **chiffre d'affaires nettement supérieur** (~458 k$ contre ~311 k$ pour New York), mais leurs **bénéfices sont très proches** (~76 k$ vs ~74 k$). New York affiche donc une **marge bénéficiaire plus élevée** : chaque dollar vendu y est plus rentable.

## 5. Client exceptionnel à New York

On identifie le client de New York qui rapporte le plus de bénéfices (et de ventes).

In [ ]:
ny = df[df["State"] == "New York"]

ny_customers = (ny.groupby("Customer Name")
                  .agg(Sales=("Sales","sum"),
                       Profit=("Profit","sum"),
                       Orders=("Order ID","nunique"))
                  .sort_values("Profit", ascending=False))

top_ny = ny_customers.head(10)
print(top_ny)

plt.figure(figsize=(12,6))
sns.barplot(x=top_ny["Profit"], y=top_ny.index, palette="crest")
plt.title("Top 10 clients de New York par bénéfice")
plt.xlabel("Bénéfice ($)")
plt.ylabel("Client")
plt.tight_layout()
plt.show()

**Observation :** **Tom Ashbrook** est le client exceptionnel de New York : il génère de très loin le bénéfice le plus élevé de l'État (~4 600 $), ce qui en fait un client à fidéliser en priorité.

## 6. Différences de rentabilité entre États

On compare le bénéfice total **et** la marge bénéficiaire moyenne par État. Certains États peuvent générer beaucoup de ventes tout en étant peu rentables (voire déficitaires).

In [ ]:
state_perf = (df.groupby("State")
                .agg(Sales=("Sales","sum"),
                     Profit=("Profit","sum"))
                .sort_values("Profit", ascending=False))
state_perf["Margin %"] = 100 * state_perf["Profit"] / state_perf["Sales"]

# États les plus rentables et les plus déficitaires
print("Top 10 États les plus rentables :")
print(state_perf.head(10))
print("\n10 États les moins rentables :")
print(state_perf.tail(10))

In [ ]:
plt.figure(figsize=(12,10))
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in state_perf["Profit"]]
sns.barplot(x=state_perf["Profit"], y=state_perf.index, palette=colors)
plt.title("Bénéfice total par État (vert = profit, rouge = perte)")
plt.xlabel("Bénéfice ($)")
plt.ylabel("État")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

**Observation :** oui, il existe de fortes différences de rentabilité. Des États comme **le Texas, l'Ohio, la Pennsylvanie et l'Illinois** sont **déficitaires** malgré des ventes importantes — souvent à cause de remises trop agressives. À l'inverse, la Californie, New York et Washington sont très rentables.

## 7. Principe de Pareto : 20 % des clients font-ils 80 % des bénéfices ?

In [ ]:
cust_profit = (df.groupby("Customer Name")["Profit"].sum()
                 .sort_values(ascending=False))

cum_profit = cust_profit.cumsum()
cum_pct    = 100 * cum_profit / cust_profit.sum()
cust_pct   = 100 * np.arange(1, len(cust_profit)+1) / len(cust_profit)

# Nombre de clients pour atteindre 80 % des bénéfices
n_80 = int((cum_pct <= 80).sum()) + 1
print(f"{n_80} clients sur {len(cust_profit)} "
      f"({100*n_80/len(cust_profit):.1f} %) génèrent 80 % des bénéfices.")

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(cust_pct, cum_pct, color="#4C72B0", linewidth=2)
plt.axhline(80, color="red", linestyle="--", label="80 % des bénéfices")
plt.axvline(20, color="green", linestyle="--", label="20 % des clients")
plt.title("Courbe de Pareto — Clients vs Bénéfices cumulés")
plt.xlabel("% des clients (classés par bénéfice décroissant)")
plt.ylabel("% cumulé des bénéfices")
plt.legend()
plt.tight_layout()
plt.show()

**Observation :** environ **20 % des clients génèrent 80 % des bénéfices** — le principe de Pareto s'applique bien ici. Ces clients à forte valeur méritent des programmes de fidélité ciblés.

⚠️ Note : les bénéfices cumulés dépassent 100 % avant de redescendre, car certains clients sont **déficitaires** (leur bénéfice cumulé négatif fait baisser la courbe en fin de classement).

## 8. Top 20 villes — ventes & bénéfices + différences de rentabilité

In [ ]:
city_perf = (df.groupby("City")
               .agg(Sales=("Sales","sum"),
                    Profit=("Profit","sum")))
city_perf["Margin %"] = 100 * city_perf["Profit"] / city_perf["Sales"]

top20_sales  = city_perf.sort_values("Sales", ascending=False).head(20)
top20_profit = city_perf.sort_values("Profit", ascending=False).head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,8))

sns.barplot(x=top20_sales["Sales"], y=top20_sales.index, palette="Blues_r", ax=axes[0])
axes[0].set_title("Top 20 villes par chiffre d'affaires")
axes[0].set_xlabel("Ventes ($)")

sns.barplot(x=top20_profit["Profit"], y=top20_profit.index, palette="Greens_r", ax=axes[1])
axes[1].set_title("Top 20 villes par bénéfice")
axes[1].set_xlabel("Bénéfice ($)")

plt.tight_layout()
plt.show()

In [ ]:
# Différences de rentabilité parmi les villes à fort chiffre d'affaires
print("Rentabilité des 20 villes au plus fort CA :")
print(top20_sales.sort_values("Margin %"))

**Observation :** New York City, Los Angeles et Seattle dominent à la fois les ventes et les bénéfices. Mais certaines villes à fort CA sont **peu rentables voire déficitaires** (ex. **Philadelphie**, **Houston**, **San Antonio**), typiquement à cause de remises élevées. Les classements ventes et bénéfices ne se recouvrent donc pas totalement.

## 9. Top 20 clients par ventes

In [ ]:
top20_cust_sales = (df.groupby("Customer Name")["Sales"].sum()
                      .sort_values(ascending=False).head(20))

plt.figure(figsize=(12,8))
sns.barplot(x=top20_cust_sales.values, y=top20_cust_sales.index, palette="rocket")
plt.title("Top 20 clients par chiffre d'affaires")
plt.xlabel("Ventes ($)")
plt.ylabel("Client")
plt.tight_layout()
plt.show()

top20_cust_sales

## 10. Courbe cumulative des ventes par client — Pareto ventes

In [ ]:
cust_sales = (df.groupby("Customer Name")["Sales"].sum()
                .sort_values(ascending=False))

cum_sales     = cust_sales.cumsum()
cum_sales_pct = 100 * cum_sales / cust_sales.sum()
cust_sales_pct = 100 * np.arange(1, len(cust_sales)+1) / len(cust_sales)

n_80_sales = int((cum_sales_pct <= 80).sum()) + 1
print(f"{n_80_sales} clients sur {len(cust_sales)} "
      f"({100*n_80_sales/len(cust_sales):.1f} %) génèrent 80 % des ventes.")

plt.figure(figsize=(12,6))
plt.plot(cust_sales_pct, cum_sales_pct, color="#8172B3", linewidth=2)
plt.axhline(80, color="red", linestyle="--", label="80 % des ventes")
plt.axvline(20, color="green", linestyle="--", label="20 % des clients")
plt.title("Courbe cumulative des ventes par client (Pareto)")
plt.xlabel("% des clients (classés par ventes décroissantes)")
plt.ylabel("% cumulé des ventes")
plt.legend()
plt.tight_layout()
plt.show()

**Observation :** pour les ventes, la concentration est un peu moins marquée que pour les bénéfices : il faut environ **35–40 % des clients pour atteindre 80 % des ventes**. Le Pareto strict 80/20 s'applique donc **mieux aux bénéfices qu'aux ventes**. Autrement dit, la valeur réelle se concentre sur les clients **rentables**, pas seulement gros acheteurs.

## Bonus — Tendance temporelle des ventes et bénéfices

In [ ]:
monthly = df.groupby("YearMonth")[["Sales","Profit"]].sum()

fig, ax = plt.subplots(figsize=(14,6))
monthly["Sales"].plot(ax=ax, label="Ventes", color="#4C72B0")
monthly["Profit"].plot(ax=ax, label="Bénéfice", color="#2ca02c")
ax.set_title("Évolution mensuelle des ventes et bénéfices")
ax.set_xlabel("Mois")
ax.set_ylabel("$")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. 🎯 Recommandations marketing

**États à privilégier :**
- **Californie & New York** — piliers du chiffre d'affaires et des bénéfices. À protéger et développer.
- **Washington** — bonne rentabilité, potentiel de croissance.
- **Attention** aux États **déficitaires** (Texas, Ohio, Pennsylvanie, Illinois) : revoir la **politique de remises** avant d'y investir en marketing.

**Villes à privilégier :**
- **New York City, Los Angeles, Seattle, San Francisco** : fort CA **et** forte rentabilité → cibles marketing prioritaires.
- Réévaluer **Philadelphie, Houston, San Antonio** : gros volumes mais pertes → réduire les remises plutôt qu'augmenter le marketing.

**Clients :**
- Appliquer le Pareto : concentrer les efforts de fidélisation sur les **~20 % de clients** générant l'essentiel des **bénéfices** (ex. **Tom Ashbrook** à NY).
- Distinguer les gros **acheteurs** des clients réellement **rentables** — cibler ces derniers.

**Levier transversal :**
- Les pertes proviennent surtout de **remises excessives**. Optimiser la stratégie de remises améliorerait la rentabilité globale sans nuire aux ventes.